# HuggingFace commit explorer

Explores dataset and model commit histories on the HuggingFace Hub to recover
the timing of contributions, together with likes and downloads, per language.

In [ ]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import re
from datetime import datetime

# Likes and downloads

In [ ]:
huggingface_languages_link = 'https://huggingface.co/languages'

In [ ]:
def extract_huggingface_languages(url):
    html_text = requests.get(url).text
    soup = BeautifulSoup(html_text, 'html.parser')
    # Initialize lists to store extracted data
    languages = []
    iso_codes = []
    datasets_links = []
    models_links = []
    datasets_numbers = []
    model_numbers = []

    # Extract data from each row of the table
    for row in soup.find_all('tr'):
        cols = row.find_all('td')
        if len(cols) > 0:  # Ensure it's not the header row
            language = cols[0].get_text(strip=True)
            iso_code = cols[1].find('code').get_text(strip=True)
            datasets_link = "https://huggingface.co" + cols[2].find('a')['href']
            models_link = "https://huggingface.co" + cols[3].find('a')['href']
            datasets_number = int(cols[2].find('a').get_text(strip=True).replace(',', ''))
            model_number = int(cols[3].find('a').get_text(strip=True).replace(',', ''))
            
            # Append data to lists
            languages.append(language)
            iso_codes.append(iso_code)
            datasets_links.append(datasets_link)
            models_links.append(models_link)
            datasets_numbers.append(datasets_number)
            model_numbers.append(model_number)

    # Create DataFrame
    df = pd.DataFrame({
        'Language': languages,
        'ISO Code': iso_codes,
        'Datasets Link': datasets_links,
        'Models Link': models_links,
        'Datasets Number': datasets_numbers,
        'Model Number': model_numbers
    })
    return df

def process_huggingface_datasets(dataset_url):
    dataset_request = requests.get(dataset_url).text
    dataset_soup = BeautifulSoup(dataset_request, 'html.parser')
    # Extract text and href attributes for all articles
    articles_data = []
    articles = dataset_soup.find_all('article')
    for article in articles:
        text = article.h4.text.strip()
        href = article.a['href']
        articles_data.append({'dataset': text, 'dataset_url': href})
        

    # Convert to DataFrame
    df = pd.DataFrame(articles_data)
    return df

In [ ]:
df = extract_huggingface_languages(huggingface_languages_link)

In [ ]:
df

### Tests on finding datasets for one language

In [ ]:
dataset_stats = process_huggingface_datasets('https://huggingface.co/datasets?language=language:ofs')

In [ ]:
dataset_stats

# Working with more datasets and more languages

In [ ]:
def find_huggingface_datasets(dataset_url, df = None):
    '''
    This function recursively retrieves dataset URLs from the Hugging Face website, starting from a given URL. 
    It iterates through all pages, extracts dataset names and URLs, and returns a list of dataset URLs.
    '''
    if df is None:
        df = pd.DataFrame(columns=['dataset', 'dataset_url'])

    dataset_request = requests.get(dataset_url).text
    dataset_soup = BeautifulSoup(dataset_request, 'html.parser')
    # Extract text and href attributes for all articles
    articles_data = []
    articles = dataset_soup.find_all('article')
    for article in articles:
        text = article.h4.text.strip()
        href = article.a['href']
        articles_data.append({'dataset': text, 'dataset_url': href})
        

    # Convert to DataFrame
    temp_df = pd.DataFrame(articles_data)
    print('first_df!')
    df = pd.concat([df, temp_df], ignore_index=True)

    sections = dataset_soup.find_all('section')[-1]
    next_page_link = sections.find('nav').find_all('a')[-1].get('href')
    if next_page_link: 
        next_page_url = f'https://huggingface.co/datasets{next_page_link}'
        return find_huggingface_datasets(next_page_url, df)
    else: 
        print('language_done!')
        return df['dataset_url'].tolist()
    

def find_earliest_commit(model_name):
    '''
    This function retrieves the commit history of a model from the Hugging Face website. 
    It identifies the earliest commit timestamp if available.
    '''
    soup = BeautifulSoup(requests.get(f'https://huggingface.co{model_name}').text, 'html.parser')
    hrefs = [a['href'] for a in soup.find_all('a', class_ = 'tab-alternate')]
    commits_href = ['https://huggingface.co' + url for url in hrefs if '/tree/main' in url]
    if commits_href:
        commits_href_history = commits_href[0].replace('tree', 'commits')
        commits_soup = BeautifulSoup(requests.get(commits_href_history).text, 'html.parser')
        time_elements = commits_soup.find_all('time')
        datetimes = [time['datetime'] for time in time_elements]
        processed_datetime = [datetime.fromisoformat(dt_str) for dt_str in datetimes]
        if processed_datetime:
            return min(processed_datetime)
    else:
        print('commits_not_found')
        return 'nothing'
    

def find_earliest_datetime_pls(lst):
    datetimes = [item for item in lst if isinstance(item, datetime)]
    earliest_datetime = min(datetimes) if datetimes else None
    return earliest_datetime

## Find the oldest commit date

Switch from the *tree/main* tab to the commit list and take the earliest date.

## Were the initial files uploaded together?

## Try on small dataset for testing

In [ ]:
df_small = df[100:105]

In [ ]:
df_small

In [ ]:
df_small['all_datasets'] = df_small['Datasets Link'].apply(lambda url: find_huggingface_datasets(url))

In [ ]:
df_small

# Try on even smaller data for date finding

In [ ]:
df_sample = df_small[0:1]

In [ ]:
df_sample['earliest_update'] = df_sample['all_datasets'].apply(lambda x: [find_earliest_commit(item) for item in x])

In [ ]:
df_sample

In [ ]:
df_sample['earliest_datetime'] = df_sample['earliest_update'].apply(find_earliest_datetime_pls)

In [ ]:
df_sample

In [ ]:
def extract_next_page_url(soup):
    sections = soup.find_all('section')[-1]
    if sections:
        next_page_link = last_section.find('nav').find_all('a')[-1].get('href')
    return next_page_link

In [ ]:
def find_data_per_language(language_url):
    language_file = requests.get(language_url)
    if language_file.status_code == 200:
        soup = BeautifulSoup(language_file.text, 'html.parser')
        next_page_url = extract_next_page_url(soup) 
        while next_page_url:
            next_file = requests.get("https://huggingface.co" + next_page_url)
            if language_file.status_code == 200:
                new_soup = BeautifulSoup(language_file.text, 'html.parser')
                return new_soup
            else:
                print('Error: Unable to fetch language URL:', language_url)
                break